# 1. Ontology Validator: Rules & Architecture Guide

This notebook uses the `validate_ontology()` function to act as a strict linter for our Semantic Ontology (`.ttl` files).

Because our ontology acts as **Configuration as Code**—controlling both the LLM's reasoning and the backend Spanner data pipelines—it must adhere to strict structural rules. The validator checks three main semantic categories and will flag issues as either **Blocking Errors** (which fail validation) or **Warnings** (best practices).

Here is exactly what the validator tests for and why.

## 1. Datatype Properties (Graph Columns / Attributes)

These properties represent the literal values and metrics in the database (e.g., employee names, dates, boolean flags). The validator ensures they are properly anchored and configured for the background ingestion pipeline.

### 🛑 Blocking Errors

* **Missing Universal Lookup Table (ULT) Flag (`ax:includeInLookup`)**
  * **What it checks:** Every property must explicitly declare `ax:includeInLookup` as either `true` or `false`.
  * **Why:** This directive controls the weekly background pipeline. The system needs to know explicitly whether to extract, embed, and upsert these values into the Spanner ULT so the AI can search for them.

* **Missing or Invalid Search Strategy (`ax:searchStrategy`)**
  * **What it checks:** If a property is marked to be included in the ULT (`includeInLookup = "true"`), it *must* declare a search strategy of either `"EXACT_OR_LIKE"` or `"SEMANTIC_VECTOR"`.
  * **Why:** Vector math is expensive and bad at exact string matching (like names or IDs). This forces the architect to explicitly define how the Resolver Agent should query the database, saving compute costs and preventing false positive matches.

* **Floating Properties (Missing `rdfs:domain` or `ax:appliesToEdge`)**
  * **What it checks:** A property cannot exist in a vacuum. It must declare an `rdfs:domain` (if it belongs to a Node/Class) OR an `ax:appliesToEdge` (if it belongs to an Object Property/Edge).
  * **Why:** Standard RDF struggles with Property Graphs where edges have columns. This strict check ensures the Planner Agent knows exactly where a column lives in the Spanner schema so it can generate accurate GQL `WHERE` clauses.

### ⚠️ Warnings

* **Missing Future-Proofing Flag (`ax:doNotEnrich`)**
  * **What it checks:** Warns if the property is missing the `doNotEnrich` directive.
  * **Why:** This is an optional feature intended for future use (allowing the pipeline to inject Top 5 DB examples directly into the LLM prompt to prevent hallucination).

* **Missing Human-Readable Label (`rdfs:label`)**
  * **What it checks:** Warns if the property lacks a label.
  * **Why:** LLMs map user intent using semantic labels. Without a label, the Planner Agent will struggle to understand what the database column actually represents.

## 2. Object Properties (Graph Edges / Relationships)

These properties represent the connections between different nodes in your Spanner graph (e.g., `hasManager`, `workedAt`).

### 🛑 Blocking Errors

* **Missing Source Node (`rdfs:domain`)**
  * **What it checks:** Unless inheriting from a parent via `rdfs:subPropertyOf`, the edge must declare where it originates.
  * **Why:** Graph traversals require a starting point.

* **Missing Destination Node (`rdfs:range`)**
  * **What it checks:** Unless inheriting from a parent via `rdfs:subPropertyOf`, the edge must declare where it terminates.
  * **Why:** Graph traversals require an endpoint.

* **Missing Physical Edge Label (`ax:gqlEdgeLabel`)**
  * **What it checks:** Every edge must declare the exact Spanner physical edge label (e.g., `"CONNECTED_TO"`).
  * **Why:** This completely eliminates hallucination. Without this, the Executor Agent will guess the GQL syntax (often incorrectly writing `-[:hasManager]->` instead of the actual Spanner DDL `-[:CONNECTED_TO]->`).

## 3. Classes (Graph Nodes / Entities)

These represent the core entities in your graph (e.g., `Person`, `Office`, `Skill`).

### ⚠️ Warnings

* **Missing Human-Readable Label (`rdfs:label`)**
  * **What it checks:** Warns if the class lacks a descriptive label.
  * **Why:** Just like Datatype properties, the LLM needs to semantically understand the entity it is querying.

## How to Run the Validator in this Notebook

Run the following code cell to validate your ontology file:

```python
from ttl_validator import validate_ontology

# Run the validator
success = validate_ontology(
    file_path="your_ontology_file.ttl", 
    namespace_uri="[https://your-company.com/ontology#](https://your-company.com/ontology#)"
)

if success:
    print("Ready for deployment!")

## The Function

In [ ]:
"""
Ontology Validator
============================
This script validates a given TTL file against the strict structural and 
architectural requirements of the Multi-Agent system. 
It ensures that all required semantic directives (ax:includeInLookup, 
ax:searchStrategy, ax:gqlEdgeLabel) are present and properly formatted 
to prevent pipeline or LLM routing failures.
"""

import sys
import argparse
import logging
import rdflib
from rdflib.namespace import RDF, OWL, RDFS

# Configure Logging (Notebook friendly: forces output to standard out so it doesn't get buried)
logging.basicConfig(level=logging.INFO, format='%(message)s', stream=sys.stdout, force=True)

def validate_ontology(file_path: str, namespace_uri: str = "https://example.com/ontology#") -> bool:
    logging.info(f"Loading and parsing ontology: {file_path}")
    g = rdflib.Graph()
    try:
        g.parse(file_path, format="turtle")
    except Exception as e:
        logging.error(f"❌ FATAL: Failed to parse TTL file. Invalid Turtle syntax.\nDetails: {e}")
        return False

    AX = rdflib.Namespace(namespace_uri)
    errors = []
    warnings = []

    def get_local_name(uri):
        if not uri: return "UNKNOWN"
        return str(uri).split('#')[-1] if '#' in str(uri) else str(uri).split('/')[-1]

    logging.info("\n--- Validating Datatype Properties (Columns/Attributes) ---")
    datatype_props = list(g.subjects(RDF.type, OWL.DatatypeProperty))
    for prop in datatype_props:
        prop_name = get_local_name(prop)
        
        # 1. Pipeline Directives Check
        include_in_lookup = g.value(prop, AX.includeInLookup)
        if include_in_lookup is None:
            errors.append(f"[DatatypeProperty: {prop_name}] Missing required directive 'ax:includeInLookup'.")
            
        do_not_enrich = g.value(prop, AX.doNotEnrich)
        if do_not_enrich is None:
            warnings.append(f"[DatatypeProperty: {prop_name}] Missing optional directive 'ax:doNotEnrich' (intended for future use).")

        # 2. Search Strategy Validation (Only if indexed)
        if include_in_lookup and str(include_in_lookup).lower() == "true":
            strategy = g.value(prop, AX.searchStrategy)
            if not strategy:
                errors.append(f"[DatatypeProperty: {prop_name}] Marked for indexing (includeInLookup=true) but missing 'ax:searchStrategy'.")
            elif str(strategy) not in ["EXACT_OR_LIKE", "SEMANTIC_VECTOR"]:
                errors.append(f"[DatatypeProperty: {prop_name}] Invalid searchStrategy '{strategy}'. Must be 'EXACT_OR_LIKE' or 'SEMANTIC_VECTOR'.")

        # 3. Structural Anchor Check (Must belong to a Node or an Edge)
        domain = g.value(prop, RDFS.domain)
        applies_to_edge = g.value(prop, AX.appliesToEdge)
        
        if not domain and not applies_to_edge:
            errors.append(f"[DatatypeProperty: {prop_name}] Floating property. Must declare 'rdfs:domain' (for nodes) OR 'ax:appliesToEdge' (for edges).")
        
        # Suggestion: Missing label
        label = g.value(prop, RDFS.label)
        if not label:
            warnings.append(f"[DatatypeProperty: {prop_name}] Missing 'rdfs:label'. LLM may struggle to understand this field.")

    logging.info("\n--- Validating Object Properties (Graph Edges) ---")
    object_props = list(g.subjects(RDF.type, OWL.ObjectProperty))
    for prop in object_props:
        prop_name = get_local_name(prop)
        
        # 1. Graph Connectivity Checks
        domain = g.value(prop, RDFS.domain)
        range_val = g.value(prop, RDFS.range)
        sub_prop = g.value(prop, RDFS.subPropertyOf)
        
        # If it's a subProperty, we assume it inherits domain and range from its parent
        if not domain and not sub_prop:
            errors.append(f"[ObjectProperty: {prop_name}] Missing 'rdfs:domain' (Source Node).")
            
        if not range_val and not sub_prop:
            errors.append(f"[ObjectProperty: {prop_name}] Missing 'rdfs:range' (Destination Node).")
            
        # 2. Spanner GQL Mapping Check
        gql_edge_label = g.value(prop, AX.gqlEdgeLabel)
        if not gql_edge_label:
            errors.append(f"[ObjectProperty: {prop_name}] Missing required directive 'ax:gqlEdgeLabel'. Executor Agent will hallucinate edge types without this.")

    logging.info("\n--- Validating Classes (Graph Nodes) ---")
    classes = list(g.subjects(RDF.type, OWL.Class))
    for cls in classes:
        cls_name = get_local_name(cls)
        
        # Suggestion: Missing label
        label = g.value(cls, RDFS.label)
        if not label and cls_name not in ["Ontology", "AnnotationProperty"]:
            warnings.append(f"[Class: {cls_name}] Missing 'rdfs:label'. LLM may struggle to understand this entity.")

    # ==========================================
    # OUTPUT REPORT
    # ==========================================
    logging.info("\n==============================================")
    logging.info("             VALIDATION REPORT                ")
    logging.info("==============================================")
    
    if not errors and not warnings:
        logging.info("✅ SUCCESS: The ontology is perfectly configured for the system!")
        return True
        
    if warnings:
        logging.warning(f"⚠️  WARNINGS ({len(warnings)}):")
        for w in warnings:
            logging.warning(f"   - {w}")
            
    if errors:
        logging.error(f"❌ ERRORS ({len(errors)}):")
        for e in errors:
            logging.error(f"   - {e}")
        logging.error("\nValidation FAILED. Please fix the above errors before deploying to the pipeline.")
        return False
    
    logging.info("\nValidation PASSED with warnings.")
    return True


if __name__ == "__main__":
    # ---------------------------------------------------------
    # NOTEBOOK SAFE-EXECUTION BLOCK
    # ---------------------------------------------------------
    # If run in a Jupyter Notebook, argparse and sys.exit() will crash the kernel.
    # This detects the environment and skips CLI parsing if we are in a notebook.
    if 'ipykernel' in sys.modules or 'IPython' in sys.modules:
        logging.info("💡 Jupyter Notebook environment detected. Skipping CLI argument parsing.")
        logging.info("👉 To run the validator, add a new cell and run: `validate_ontology('path_to_your_file.ttl', namespace_uri='https://example.com/ontology#')`")
    else:
        parser = argparse.ArgumentParser(description="Validate Ontology files.")
        parser.add_argument("file", help="Path to the .ttl file to validate.")
        parser.add_argument("--namespace", default="https://example.com/ontology#", 
                            help="The base namespace URI for custom directives.")
        
        args = parser.parse_args()
        
        success = validate_ontology(args.file, args.namespace)
        sys.exit(0 if success else 1)

## Run Validator

In [ ]:
ontology_path = '../spanner/sample_team_agent_graph/ontology.ttl'
validate_ontology(ontology_path, namespace_uri='https://ontology.mock_corporation.com/teamagent#')

# 2. Ontology Compiler Test

In [1]:
import sys
import os

# 1. Add the parent directory (root) to the Python path
sys.path.append(os.path.abspath('..'))

# 2. Now Python can find the 'utilities' folder at the root level!
from utilities.ontology_compiler import OntologyCompiler

# Use it
ontology_path = '../spanner/sample_team_agent_graph/ontology.ttl'

# 3. Make sure to pass the path variable here, not the hardcoded string
compiler = OntologyCompiler(ontology_path)
summary = compiler.compile_summary()
print(summary)

ModuleNotFoundError: No module named 'utilities.ontology_compiler'